# 01 — Data acquisition and audit

Phase 1 of the Cape Town service-request resolution classifier.

This notebook prepares the project and, when a raw file is present, describes that file. It does not clean the extract, fit a preprocessor, or train a model.

Run it from the top after placing the original download in `data/raw/`. If the file is missing, the notebook stops at the manual download steps. It does not create a substitute dataset.


## 1. Problem definition

When a City of Cape Town municipal service request is logged, service departments need an early view of how it is likely to end. The classifier predicts one of three outcomes:

- **Resolved Fast**
- **Resolved Slow**
- **Unresolved**

The prediction must use only information that is known at logging. Later notebooks will compare models with macro-F1 as the primary metric, and they will pay particular attention to recall for Resolved Slow and Unresolved. Those models are out of scope here.


## 2. Dataset

The project plan asks for City of Cape Town SAP C3 Notifications from 2023 until 18 September 2026.

The dataset page used here is:

https://odp-cctegis.opendata.arcgis.com/datasets/90711027d33940b5a06e96ad8a8f7ede_0/explore

The published title is **Service Requests 2023 until 06 October 2026 (Deprecated)**. The City describes the layer as frozen. A feature-service query on 16 October 2026 counted 5,372,466 rows. `Created_On_Date` runs from 1 January 2023 through 6 October 2026 (Johannesburg midnight), which is later than the plan's end date. Field names, null counts, the licence, and the query notes are in `reports/data_source.md`.

No local copy is in `data/raw/`. On that same day the portal CSV export was still being generated, so this notebook does not invent a file or a row-level audit.


## 3. Leakage and the observation window

The label will later be built from **Created On Date** and **Completed Date**. **Completed Date** is not a model feature. **Changed on** can reflect updates after logging, so it is excluded as well.

An open request that was logged only a few days before the end of the study can look Unresolved simply because not enough time has passed. Those rows need an observation window. The fast/slow boundary also has to be chosen from the resolution-time distribution.

Both thresholds are unset in Phase 1 (`None` in `src/config.py`). This notebook does not choose them and does not build the final label column.


## 4. Setup

The next cell finds the project root, imports the shared modules, and sets the random seed. PyTorch is imported only to confirm the chosen neural-network framework loads. No network is defined or trained.


In [1]:
from pathlib import Path
import sys

import matplotlib
import pandas as pd
import torch

# Headless execution can save figures later. JupyterLab still displays outputs.
matplotlib.use("Agg", force=False)

cwd = Path.cwd().resolve()
project_root = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "src" / "config.py").is_file():
        project_root = candidate
        break
if project_root is None:
    raise FileNotFoundError(
        "Could not find src/config.py. Start JupyterLab from the project folder."
    )
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src import config
from src.data_loader import (
    RawDataNotFoundError,
    audit_frame,
    list_raw_files,
    load_raw_table,
    manual_download_instructions,
    resolve_raw_file,
)
from src.preprocessing import planned_column_report, select_model_features

config.set_random_seeds(config.RANDOM_SEED)
print("Project root:", project_root)
print("Random seed:", config.RANDOM_SEED)
print("PyTorch:", torch.__version__)
print("pandas:", pd.__version__)
print("Target labels:", config.TARGET_LABELS)
print("Fast/slow threshold (days):", config.FAST_SLOW_THRESHOLD_DAYS)
print("Observation window (days):", config.OBSERVATION_WINDOW_DAYS)
print("Source page URL:", config.SOURCE_PAGE_URL)
print("Planned collection period:", config.PLANNED_COLLECTION_PERIOD)


Project root: /workspace
Random seed: 42
PyTorch: 2.14.1+cpu
pandas: 3.0.6
Target labels: ('Resolved Fast', 'Resolved Slow', 'Unresolved')
Fast/slow threshold (days): None
Observation window (days): None
Source page URL: https://odp-cctegis.opendata.arcgis.com/datasets/90711027d33940b5a06e96ad8a8f7ede_0/explore
Planned collection period: 2023 until 18 September 2026


## 5. Project folders

These paths are where later notebooks will read and write. This cell checks that the folders exist. It does not create data.


In [2]:
for path in config.required_directories():
    print(f"{'ok' if path.is_dir() else 'MISSING':7}  {path.relative_to(config.PROJECT_ROOT)}")

print("Raw files:", [path.name for path in list_raw_files()])


ok       data/raw
ok       data/interim
ok       data/processed
ok       notebooks
ok       models
ok       app
ok       results/figures
ok       results/tables
ok       reports
ok       tests
ok       src
Raw files: []


## 6. Locate the raw extract

If `data/raw/` is empty, the cell prints the manual steps and sets `data_available` to false. No sample rows are invented. If several files are present, the cell also stops, because the loader will not choose one by itself.


In [3]:
data_available = False
raw_path = None
try:
    raw_path = resolve_raw_file()
except (RawDataNotFoundError, ValueError) as error:
    print(error)
else:
    data_available = True
    print("Raw file found:", raw_path)
    print("The file is read only. This notebook does not overwrite it.")


No raw file is in data/raw yet.

Dataset page:
https://odp-cctegis.opendata.arcgis.com/datasets/90711027d33940b5a06e96ad8a8f7ede_0/explore

Published title: Service Requests 2023 until  06 October 2026 (Deprecated)
Portal owner: City of Cape Town
Licence link on the item: https://www.capetown.gov.za/General/Terms-of-use-open-data

On 16 October 2026 the portal CSV export was still being generated, so this
repository has no local copy. Counts, fields, and date limits already read
from the feature service are in reports/data_source.md.

1. Open the dataset page above and download the original file when the portal offers it.
2. Save that file, unchanged, in data/raw/. CSV is the preferred format.
3. If you save more than one file, set DATA_RAW_FILE in .env to the file this audit should read.
4. Do not commit the raw file. data/raw/ is gitignored except for .gitkeep.
5. Do not trim the raw file to the project-plan end date of 18 September 2026.
   The published Created On Date values conti

## 7. Audit the file that was actually loaded

The service field list is already written in `reports/data_source.md`. This cell does not repeat those counts as if they came from a local file.

It runs the row, column, dtype, and missingness audit only when a raw file was found. It then checks the date columns. The project brief uses `Created On Date`, `Completed Date`, and `Changed on`. The feature service uses `Created_On_Date`, `Completed_Date`, and `Changed_on`. Either spelling counts as a match. Any other spelling is left unmatched. Columns are not renamed.


In [4]:
if not data_available:
    print("Local-file audit skipped. data/raw has no extract, so this cell has no file-level column names or counts.")
    print("Service-level fields and counts already recorded on 16 October 2026 are in reports/data_source.md.")
    print("After the portal file is saved in data/raw/, run this notebook again from the top.")
else:
    frame = load_raw_table(raw_path)
    summary = audit_frame(frame)
    print(f"Rows: {summary['row_count']}")
    print(f"Columns: {summary['column_count']}")
    column_table = pd.DataFrame(
        {
            "column": summary["columns"],
            "dtype": [summary["dtypes"][name] for name in summary["columns"]],
            "missing_count": [summary["missing_count"][name] for name in summary["columns"]],
            "missing_percent": [round(summary["missing_percent"][name], 2) for name in summary["columns"]],
        }
    )
    display(column_table)
    name_report = planned_column_report(frame.columns)
    display(name_report)
    kept = select_model_features(frame.columns)
    removed = [name for name in frame.columns if name not in kept]
    print("Columns removed by the current leakage list:")
    print(removed if removed else "(none of the known leakage names were found)")
    print(
        "Columns still in the feature list still need a manual leakage review. "
        "This notebook does not drop them automatically."
    )
    table_path = config.TABLES_DIR / "01_column_audit.csv"
    column_table.to_csv(table_path, index=False)
    print("Wrote", table_path.relative_to(config.PROJECT_ROOT))


Local-file audit skipped. data/raw has no extract, so this cell has no file-level column names or counts.
Service-level fields and counts already recorded on 16 October 2026 are in reports/data_source.md.
After the portal file is saved in data/raw/, run this notebook again from the top.


## 8. What this phase does not do

- It does not estimate the fast/slow threshold or the observation window.
- It does not impute, encode, scale, or split the data.
- It does not fit a dummy baseline, a logistic regression, a tree, or a neural network.
- It does not treat the feature-service summary as a substitute for the local file audit.

## Next step

When the portal finishes the CSV export, save the original file in `data/raw/` and re-run this notebook from the top. Compare its columns with `reports/data_source.md`. Do not start cleaning until that local audit has been printed. Exploratory data analysis is the following notebook, and it is not created yet.
